# 01 — Raw Data Exploration
Connects to PostgreSQL and explores the raw schema tables before any transformation.
Covers row counts, data types, null checks and sample records for customers, products and orders.

In [11]:
import subprocess
subprocess.run(["pip", "install", "psycopg2-binary", "pymongo", "minio"], check=True)
print("Dependencies installed!")

Dependencies installed!


In [12]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

# Build SQLAlchemy connection string from Docker environment variables
PG_HOST     = os.environ["POSTGRES_HOST"]
PG_PORT     = os.environ["POSTGRES_PORT"]
PG_DB       = os.environ["POSTGRES_DB"]
PG_USER     = os.environ["POSTGRES_USER"]
PG_PASSWORD = os.environ["POSTGRES_PASSWORD"]

# SQLAlchemy engine — preferred by pandas for read_sql()
engine = create_engine(
    f"postgresql://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"
)

print("Connected to PostgreSQL via SQLAlchemy!")

Connected to PostgreSQL via SQLAlchemy!


## Raw Schema Overview
Check what tables exist in the raw schema and how many rows each has.

In [13]:
# Query row counts for all three raw tables
query = """
SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM raw.customers
UNION ALL
SELECT 'products',                 COUNT(*)               FROM raw.products
UNION ALL
SELECT 'orders',                   COUNT(*)               FROM raw.orders;
"""

df_counts = pd.read_sql(query, engine)
df_counts

,table_name,row_count
0,customers,282
1,products,88
2,orders,37891


## Customers — Raw Data Profile

In [14]:
# Load raw customers into a DataFrame
df_customers = pd.read_sql("SELECT * FROM raw.customers;", engine)

print(f"Shape: {df_customers.shape}")
print(f"\nData types:\n{df_customers.dtypes}")
print(f"\nNull counts:\n{df_customers.isnull().sum()}")
df_customers.head()

Shape: (282, 7)

Data types:
customer_id            object
name                   object
email                  object
phone                  object
city                   object
batch_id                int64
created_at     datetime64[ns]
dtype: object

Null counts:
customer_id    0
name           0
email          0
phone          0
city           0
batch_id       0
created_at     0
dtype: int64


,customer_id,name,email,phone,city,batch_id,created_at
0,cust57705,Jack Owens,thernandez@example.net,877.553.8097,Beasleyborough,1,2026-04-18 03:26:28.322
1,cust27497,Eric Clark,jason94@example.org,808.482.0948,Howardberg,1,2026-04-18 03:26:28.317
2,cust67400,Mark Ballard,huntlaurie@example.org,5662555334,West Kelly,1,2026-04-18 03:26:28.319
3,cust24724,John Conley,wilkersonfelicia@example.org,(955)933-0606x9343,Dwaynefort,1,2026-04-18 03:26:28.318
4,cust83487,Steven Andrews,ubryant@example.com,001-400-260-4088,Ashleyborough,1,2026-04-18 03:26:28.321


## Products — Raw Data Profile

In [15]:
df_products = pd.read_sql("SELECT * FROM raw.products;", engine)

print(f"Shape: {df_products.shape}")
print(f"\nData types:\n{df_products.dtypes}")
print(f"\nNull counts:\n{df_products.isnull().sum()}")
df_products.head()

Shape: (88, 6)

Data types:
product_id              object
product_name            object
category                object
price                  float64
batch_id                 int64
created_at      datetime64[ns]
dtype: object

Null counts:
product_id      0
product_name    0
category        0
price           0
batch_id        0
created_at      0
dtype: int64


,product_id,product_name,category,price,batch_id,created_at
0,PROD9311,Yoga Mat,Sports & Fitness,14110.55,4,2026-04-18 03:37:30.534
1,PROD8545,Resistance Bands,Sports & Fitness,1098.59,4,2026-04-18 03:37:30.534
2,PROD7597,Cookbook,Books,10672.45,4,2026-04-18 03:37:30.534
3,PROD2632,Bed Sheets,Home & Kitchen,11770.66,4,2026-04-18 03:37:30.534
4,PROD6019,Smart Watch,Electronics,11784.85,4,2026-04-18 03:37:30.534


## Orders — Raw Data Profile

In [16]:
df_orders = pd.read_sql("SELECT * FROM raw.orders;", engine)

print(f"Shape: {df_orders.shape}")
print(f"\nData types:\n{df_orders.dtypes}")
print(f"\nNull counts:\n{df_orders.isnull().sum()}")
df_orders.head()

Shape: (37891, 11)

Data types:
order_id                   object
customer_id                object
product_id                 object
quantity                    int64
unit_price                float64
total_price               float64
status                     object
payment_method             object
shipping_method            object
batch_id                    int64
created_at         datetime64[ns]
dtype: object

Null counts:
order_id           0
customer_id        0
product_id         0
quantity           0
unit_price         0
total_price        0
status             0
payment_method     0
shipping_method    0
batch_id           0
created_at         0
dtype: int64


,order_id,customer_id,product_id,quantity,unit_price,total_price,status,payment_method,shipping_method,batch_id,created_at
0,ORD55633,cust85698,PROD4207,7,306.69,2146.83,pending,bank_transfer,pickup,4,2026-04-18 03:37:30.559
1,ORD75449,cust26604,PROD7597,7,425.53,2978.71,returned,credit_card,express,4,2026-04-18 03:37:30.559
2,ORD67926,cust46901,PROD9311,1,269.97,269.97,delivered,bank_transfer,express,4,2026-04-18 03:37:30.559
3,ORD37896,cust63670,PROD7597,9,21.14,190.26,delivered,debit_card,express,4,2026-04-18 03:37:30.559
4,ORD25737,cust11698,PROD8545,1,451.29,451.29,delivered,debit_card,pickup,4,2026-04-18 03:37:30.559


## Summary — Data Quality Observations

In [17]:
print("=== RAW DATA SUMMARY ===\n")
print(f"Customers : {len(df_customers):,} rows | Nulls: {df_customers.isnull().sum().sum()}")
print(f"Products  : {len(df_products):,} rows  | Nulls: {df_products.isnull().sum().sum()}")
print(f"Orders    : {len(df_orders):,} rows | Nulls: {df_orders.isnull().sum().sum()}")

print("\n=== KNOWN DATA QUALITY ISSUES ===")
print("1. customers.created_at — NULL for early batches due to 'creatred_at' typo")
print("2. products.batch_id    — NULL for early batches due to 'batch-id' hyphen typo")
print("3. These will be handled in the staging transformation notebooks")

=== RAW DATA SUMMARY ===

Customers : 282 rows | Nulls: 0
Products  : 88 rows  | Nulls: 0
Orders    : 37,891 rows | Nulls: 0

=== KNOWN DATA QUALITY ISSUES ===
1. customers.created_at — NULL for early batches due to 'creatred_at' typo
2. products.batch_id    — NULL for early batches due to 'batch-id' hyphen typo
3. These will be handled in the staging transformation notebooks
